In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from constants import FILE

In [11]:
path = Path()
file_path = path.absolute() / 'out' / FILE

In [12]:
EXCLUDED_WEIGHTS = [
    '[1, 0, 0, 0, 0]',
    '[0, 1, 0, 0, 0]',
    '[0, 0, 1, 0, 0]',
    '[0, 0, 0, 1, 0]',
    '[0, 0, 0, 0, 1]',
]

In [14]:
df = pd.read_excel(file_path, engine='openpyxl')

In [15]:
df = df[~df["weight"].isin(EXCLUDED_WEIGHTS)].copy()

In [16]:
df = df[~df['instancia'].isna()]

In [17]:
hash_list = df["weight_hash"].unique()

def make_weight_hash_map_from_list(hash_list, start_at=1):
    return {h: rf"$w_{{{i}}}$" for i, h in enumerate(hash_list, start=start_at)}

hash_map = make_weight_hash_map_from_list(hash_list)

In [18]:
df["weight_label"] = df["weight_hash"].map(hash_map).fillna(df["weight_hash"])

In [19]:
df["gap"] = df["gap"].clip(lower=0)

In [20]:
# =========================
# 1) Preparação dos dados
# =========================
# Troque "df" pelo nome do seu DataFrame, se necessário
df["gap"] = df["gap"].clip(lower=0)
base = df.copy()

cols = ["instancia", "clientes", "classe", "gap"]
faltantes = [c for c in cols if c not in base.columns]
if faltantes:
    raise ValueError(f"Colunas ausentes no DataFrame: {faltantes}")

dados = base[cols].copy()
dados["clientes"] = pd.to_numeric(dados["clientes"], errors="coerce")
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")
dados = dados.dropna(subset=["clientes", "classe", "gap"])

# Ajustes de tipo
dados["clientes"] = dados["clientes"].astype(int)
dados["classe"] = dados["classe"].astype(str)


dados["gap_pct"] = dados["gap"] * 100

# =========================
# 2) Resumo estatístico
# =========================
resumo = (
    dados.groupby(["clientes", "classe"], as_index=False)
    .agg(
        n_execucoes=("gap_pct", "size"),
        n_instancias=("instancia", "nunique"),
        gap_medio_pct=("gap_pct", "mean"),
        gap_mediano_pct=("gap_pct", "median"),
        gap_std_pct=("gap_pct", "std"),
        gap_p90_pct=("gap_pct", lambda s: s.quantile(0.90)),
        gap_min_pct=("gap_pct", "min"),
        gap_max_pct=("gap_pct", "max"),
    )
    .sort_values(["clientes", "classe"])
)

display(resumo.round(3))

# Tabelas dinâmicas
tabela_media = dados.pivot_table(
    index="clientes", columns="classe", values="gap_pct", aggfunc="mean"
).sort_index()

tabela_qtd = dados.pivot_table(
    index="clientes", columns="classe", values="gap_pct", aggfunc="size", fill_value=0
).sort_index()

display(tabela_media.round(3))
display(tabela_qtd)

# =========================
# 3) Plotly - Boxplot
# =========================
fig_box = px.box(
    dados.sort_values("clientes"),
    x="clientes",
    y="gap_pct",
    color="classe",
    points="outliers",
    labels={
        "clientes": "Quantidade de clientes",
        "gap_pct": "GAP (%)",
        "classe": "Classe",
    },
    title="Distribuição do GAP (%) por quantidade de clientes e classe",
)

fig_box.update_layout(
    boxmode="group",
    legend_title_text="Classe",
    template="plotly_white"
)
fig_box.show()

# =========================
# 4) Plotly - Heatmap (média)
# =========================
fig_heat = px.imshow(
    tabela_media,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="YlOrRd",
    labels=dict(x="Classe", y="Quantidade de clientes", color="GAP médio (%)"),
    title="Heatmap da média do GAP (%) - clientes x classe",
)

fig_heat.update_layout(template="plotly_white")
fig_heat.show()

,clientes,classe,n_execucoes,n_instancias,gap_medio_pct,gap_mediano_pct,gap_std_pct,gap_p90_pct,gap_min_pct,gap_max_pct
0,5,1.0,3588,10,4.306529e+73,0.022,2.841909e+74,7.075,0.000,2.887584e+75
1,5,2.0,3456,10,1.251000e+00,0.010,3.395000e+00,4.123,0.000,3.593700e+01
2,5,3.0,3528,10,2.127000e+00,0.024,4.630000e+00,7.077,0.000,5.953300e+01
3,5,4.0,2592,10,4.766000e+00,0.066,7.326000e+00,14.950,0.000,4.154400e+01
4,10,1.0,3600,10,5.765000e+01,62.246,2.127700e+01,79.601,0.040,8.981000e+01
5,10,2.0,3600,10,5.617700e+01,61.690,2.193200e+01,78.846,0.014,9.213500e+01
6,10,3.0,3600,10,5.607600e+01,61.476,2.184800e+01,78.956,0.008,9.262900e+01
7,10,4.0,3600,10,6.624300e+01,70.980,1.881900e+01,83.877,0.082,9.163200e+01


classe,1.0,2.0,3.0,4.0
clientes,,,,
5,4.306529e+73,1.251,2.127,4.766
10,5.765000e+01,56.177,56.076,66.243


classe,1.0,2.0,3.0,4.0
clientes,,,,
5,3588,3456,3528,2592
10,3600,3600,3600,3600


In [21]:
# =========================
# 1) Preparação dos dados
# =========================
# Troque "df" pelo nome do seu DataFrame, se necessário
base = df.copy()

required_cols = ["instancia", "alpha", "clientes", "classe", "weight_label", "gap"]
missing = [c for c in required_cols if c not in base.columns]
if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

dados = base[required_cols].copy()

dados["alpha"] = pd.to_numeric(dados["alpha"], errors="coerce")
dados["clientes"] = pd.to_numeric(dados["clientes"], errors="coerce")
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")

dados = dados.dropna(subset=["alpha", "clientes", "classe", "weight_label", "gap"])

dados["clientes"] = dados["clientes"].astype(int)
dados["classe"] = dados["classe"].astype(str)
dados["weight_label"] = dados["weight_label"].astype(str)

# Regra pedida: gap negativo vira 0
dados["gap"] = dados["gap"].clip(lower=0)

# Converte para percentual se estiver em fração
if dados["gap"].abs().max() <= 1.5:
    dados["gap_pct"] = dados["gap"] * 100
else:
    dados["gap_pct"] = dados["gap"]

# =========================
# 2) Resumo da análise
# =========================
resumo = (
    dados.groupby(["alpha", "clientes", "classe", "weight_label"], as_index=False)
    .agg(
        n_execucoes=("gap_pct", "size"),
        n_instancias=("instancia", "nunique"),
        gap_medio_pct=("gap_pct", "mean"),
        gap_mediano_pct=("gap_pct", "median"),
        gap_std_pct=("gap_pct", "std"),
        gap_p90_pct=("gap_pct", lambda s: s.quantile(0.90)),
        gap_min_pct=("gap_pct", "min"),
        gap_max_pct=("gap_pct", "max"),
    )
    .sort_values(["alpha", "clientes", "classe", "weight_label"])
)

display(resumo.round(3))

# Tabela dinâmica útil para inspeção
tabela_media = resumo.pivot_table(
    index=["alpha", "clientes"],
    columns=["classe", "weight_label"],
    values="gap_medio_pct"
).sort_index()

display(tabela_media.round(3))

# =========================
# 3) Plotly - visão geral com alpha no slider
# =========================
resumo["alpha_frame"] = resumo["alpha"].map(lambda x: f"{x:g}")

fig_line = px.line(
    resumo,
    x="clientes",
    y="gap_medio_pct",
    color="classe",
    line_dash="weight_label",
    markers=True,
    animation_frame="alpha_frame",
    hover_data={
        "n_execucoes": True,
        "n_instancias": True,
        "gap_mediano_pct": ":.2f",
        "gap_p90_pct": ":.2f",
        "gap_std_pct": ":.2f",
    },
    labels={
        "clientes": "Quantidade de clientes",
        "gap_medio_pct": "GAP médio (%)",
        "classe": "Classe",
        "weight_label": "Weight label",
        "alpha_frame": "Alpha",
    },
    title="GAP médio (%) por clientes, classe e weight_label (slider por alpha)",
)

fig_line.update_layout(template="plotly_white", legend_title_text="Classe")
fig_line.show()

# =========================
# 4) Função de heatmap (alpha + weight_label específicos)
# =========================
def plot_heatmap(alpha_sel, weight_sel):
    corte = resumo[
        (resumo["alpha"] == alpha_sel) &
        (resumo["weight_label"] == str(weight_sel))
    ]

    if corte.empty:
        print(f"Sem dados para alpha={alpha_sel} e weight_label={weight_sel}")
        return

    pivot = (
        corte.pivot(index="clientes", columns="classe", values="gap_medio_pct")
        .sort_index()
    )

    fig_heat = px.imshow(
        pivot,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="YlOrRd",
        labels={
            "x": "Classe",
            "y": "Quantidade de clientes",
            "color": "GAP médio (%)",
        },
        title=f"Heatmap do GAP médio (%) | alpha={alpha_sel} | weight_label={weight_sel}",
    )
    fig_heat.update_layout(template="plotly_white")
    fig_heat.show()

# Exemplo de uso:
plot_heatmap(alpha_sel=resumo["alpha"].iloc[0], weight_sel=resumo["weight_label"].iloc[0])

,alpha,clientes,classe,weight_label,n_execucoes,n_instancias,gap_medio_pct,gap_mediano_pct,gap_std_pct,gap_p90_pct,gap_min_pct,gap_max_pct
0,0.01,5,1.0,$w_{1}$,297,10,7.810412e+71,0.007,3.870186e+72,0.046,0.000,2.263004e+73
1,0.01,5,1.0,$w_{2}$,297,10,9.796223e+71,0.006,4.872171e+72,0.045,0.000,2.887584e+73
2,0.01,5,1.0,$w_{3}$,303,10,7.003329e+71,0.036,3.587593e+72,0.121,0.000,2.254109e+73
3,0.01,5,1.0,$w_{4}$,297,10,7.801952e+71,0.008,3.904963e+72,0.078,0.000,2.359135e+73
4,0.01,5,1.0,$w_{5}$,303,10,7.988435e+71,0.013,4.002500e+72,0.110,0.000,2.369535e+73
...,...,...,...,...,...,...,...,...,...,...,...,...
91,0.99,10,4.0,$w_{2}$,300,10,4.950000e-01,0.755,3.680000e-01,0.849,0.001,8.950000e-01
92,0.99,10,4.0,$w_{3}$,300,10,7.830000e-01,0.803,7.800000e-02,0.866,0.544,9.110000e-01
93,0.99,10,4.0,$w_{4}$,300,10,7.000000e-01,0.746,1.400000e-01,0.846,0.209,8.950000e-01
94,0.99,10,4.0,$w_{5}$,300,10,6.230000e-01,0.643,1.460000e-01,0.794,0.235,8.230000e-01


classe                   1.0                                            \
weight_label         $w_{1}$       $w_{2}$       $w_{3}$       $w_{4}$   
alpha clientes                                                           
0.01  5         7.810412e+71  9.796223e+71  7.003329e+71  7.801952e+71   
      10        5.280000e-01  6.240000e-01  6.800000e-01  6.150000e-01   
0.99  5         6.038048e+70  1.005185e+71  7.090470e+70  8.041406e+70   
      10        4.150000e-01  3.820000e-01  6.730000e-01  6.190000e-01   

classe                                         2.0                          \
weight_label         $w_{5}$       $w_{6}$ $w_{1}$ $w_{2}$ $w_{3}$ $w_{4}$   
alpha clientes                                                               
0.01  5         7.988435e+71  6.883872e+71   0.018   0.018   0.033   0.023   
      10        5.460000e-01  6.560000e-01   0.537   0.632   0.682   0.627   
0.99  5         6.904251e+70  5.983847e+70   0.000   0.000   0.000   0.004   
      10        5.050000e-01  6.750000e-01   0.457   0.365   0.628   0.551   

classe          ...     3.0                             4.0                  \
weight_label    ... $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$ $w_{1}$ $w_{2}$ $w_{3}$   
alpha clientes  ...                                                           
0.01  5         ...   0.053   0.031   0.029   0.055   0.068   0.064   0.119   
      10        ...   0.653   0.634   0.523   0.650   0.639   0.669   0.690   
0.99  5         ...   0.001   0.002   0.003   0.012   0.006   0.006   0.009   
      10        ...   0.671   0.618   0.466   0.646   0.567   0.495   0.783   

classe                                  
weight_label   $w_{4}$ $w_{5}$ $w_{6}$  
alpha clientes                          
0.01  5          0.094   0.062   0.128  
      10         0.659   0.598   0.746  
0.99  5          0.001   0.010   0.004  
      10         0.700   0.623   0.780  

[4 rows x 24 columns]

In [22]:
# Assume que df, pd e px já estão disponíveis no notebook

base = df.copy()

# Usa weight_label; fallback para weigh_label se vier com esse nome
peso_col = "weight_label" if "weight_label" in base.columns else "weigh_label"
if peso_col not in base.columns:
    raise ValueError("Coluna de peso não encontrada: 'weight_label' (ou 'weigh_label').")

dados = base[[peso_col, "gap"]].copy()
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")
dados = dados.dropna(subset=[peso_col, "gap"])

# Gap negativo vira zero
dados["gap"] = dados["gap"].clip(lower=0)

# Converte para percentual se necessário
if dados["gap"].abs().max() <= 1.5:
    dados["gap_pct"] = dados["gap"] * 100
else:
    dados["gap_pct"] = dados["gap"]

dados[peso_col] = dados[peso_col].astype(str)

# Remove extremos por grupo (IQR por weight_label)
iqr_stats = (
    dados.groupby(peso_col)["gap_pct"]
    .quantile([0.25, 0.75]).unstack()
    .rename(columns={0.25: "q1", 0.75: "q3"})
)
iqr_stats["iqr"] = iqr_stats["q3"] - iqr_stats["q1"]
iqr_stats["lim_inf"] = iqr_stats["q1"] - 1.5 * iqr_stats["iqr"]
iqr_stats["lim_sup"] = iqr_stats["q3"] + 1.5 * iqr_stats["iqr"]

dados_aux = dados.join(iqr_stats[["lim_inf", "lim_sup"]], on=peso_col)
dados_sem_extremos = dados_aux[
    (dados_aux["gap_pct"] >= dados_aux["lim_inf"]) &
    (dados_aux["gap_pct"] <= dados_aux["lim_sup"])
].drop(columns=["lim_inf", "lim_sup"])

# Resumo
resumo = (
    dados_sem_extremos.groupby(peso_col, as_index=False)
    .agg(
        n_execucoes=("gap_pct", "size"),
        gap_medio_pct=("gap_pct", "mean"),
        gap_mediano_pct=("gap_pct", "median"),
        gap_std_pct=("gap_pct", "std"),
        gap_p90_pct=("gap_pct", lambda s: s.quantile(0.90)),
        gap_min_pct=("gap_pct", "min"),
        gap_max_pct=("gap_pct", "max"),
    )
    .sort_values(peso_col)
)

display(resumo.round(3))

# Boxplot sem extremos
fig_box = px.box(
    dados_sem_extremos,
    x=peso_col,
    y="gap_pct",
    points=False,
    labels={peso_col: "weight_label", "gap_pct": "GAP (%)"},
    title="Distribuição do GAP (%) por weight_label (sem valores extremos - IQR)",
)
fig_box.update_layout(template="plotly_white")
fig_box.show()

,weight_label,n_execucoes,gap_medio_pct,gap_mediano_pct,gap_std_pct,gap_p90_pct,gap_min_pct,gap_max_pct
0,$w_{1}$,4566,0.273,0.129,0.284,0.655,0.0,0.916
1,$w_{2}$,4566,0.280,0.043,0.325,0.755,0.0,0.895
2,$w_{3}$,4578,0.373,0.265,0.354,0.821,0.0,0.926
3,$w_{4}$,4566,0.340,0.346,0.325,0.755,0.0,0.900
4,$w_{5}$,4578,0.284,0.117,0.297,0.709,0.0,0.921
5,$w_{6}$,4566,0.375,0.421,0.342,0.800,0.0,0.913


In [23]:
dados_aux

,weight_label,gap,gap_pct,lim_inf,lim_sup
0,$w_{1}$,0.000100,0.000100,-0.836648,1.395001
1,$w_{1}$,0.000100,0.000100,-0.836648,1.395001
2,$w_{1}$,0.000100,0.000100,-0.836648,1.395001
3,$w_{1}$,0.000100,0.000100,-0.836648,1.395001
4,$w_{1}$,0.000100,0.000100,-0.836648,1.395001
...,...,...,...,...,...
51907,$w_{4}$,0.793852,0.793852,-0.982326,1.637784
51908,$w_{4}$,0.793852,0.793852,-0.982326,1.637784
51909,$w_{4}$,0.793852,0.793852,-0.982326,1.637784
51910,$w_{4}$,0.793852,0.793852,-0.982326,1.637784


In [24]:
dados = base[["weight_label", "gap"]].copy()
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")
dados["gap_pct"] = dados["gap"] * 100
dados = dados.dropna(subset=["weight_label", "gap"])

iqr_stats = (
    dados.groupby("weight_label")["gap_pct"]
    .quantile([0.25, 0.75]).unstack()
    .rename(columns={0.25: "q1", 0.75: "q3"})
)
iqr_stats["iqr"] = iqr_stats["q3"] - iqr_stats["q1"]
iqr_stats["lim_inf"] = iqr_stats["q1"] - 1.5 * iqr_stats["iqr"]
iqr_stats["lim_sup"] = iqr_stats["q3"] + 1.5 * iqr_stats["iqr"]

dados_aux = dados.join(iqr_stats[["lim_inf", "lim_sup"]], on="weight_label")


# Marca outliers pelo critério IQR
dados_aux["is_outlier"] = (
    (dados_aux["gap_pct"] < dados_aux["lim_inf"]) |
    (dados_aux["gap_pct"] > dados_aux["lim_sup"])
)

# ---- Resumo geral (absoluto e %) ----
total_geral = len(dados_aux)
outliers_geral = int(dados_aux["is_outlier"].sum())
pct_outliers_geral = (100 * outliers_geral / total_geral) if total_geral else 0.0

print(f"Outliers (geral): {outliers_geral} de {total_geral} ({pct_outliers_geral:.2f}%)")

# ---- Resumo por weight_label (absoluto e %) ----
resumo_outliers = (
    dados_aux.groupby("weight_label", as_index=False)
    .agg(
        total_registros=("is_outlier", "size"),
        qtd_outliers=("is_outlier", "sum"),
    )
)

resumo_outliers["pct_outliers"] = (
    100 * resumo_outliers["qtd_outliers"] / resumo_outliers["total_registros"]
)

display(
    resumo_outliers.sort_values("qtd_outliers", ascending=False).round(2)
)

# Mantém somente não-outliers
dados_sem_extremos = dados_aux[
    ~dados_aux["is_outlier"]
].drop(columns=["lim_inf", "lim_sup", "is_outlier"])

Outliers (geral): 144 de 27564 (0.52%)


,weight_label,total_registros,qtd_outliers,pct_outliers
0,$w_{1}$,4590,24,0.52
1,$w_{2}$,4590,24,0.52
2,$w_{3}$,4602,24,0.52
3,$w_{4}$,4590,24,0.52
4,$w_{5}$,4602,24,0.52
5,$w_{6}$,4590,24,0.52


In [25]:
df.columns

Index(['time', 'file', 'hash_file', 'weight_hash', 'weight', 'FO', 'gap',
       'solver_time', 'f1', 'f2', 'f3', 'f4', 'f5', 'p1', 'p2', 'p3', 'p4',
       'p5', 'epsilon', 'total_production', 'total_inventory', 'total_setup',
       'total_delivered', 'csetup', 'cprod', 'instancia', 'clientes',
       'produtos', 'veiculos', 'periodos', 'seeds', 'classe', 'new_f1_target',
       'new_f2_target', 'new_f3_target', 'new_f4_target', 'new_f5_target',
       'hash_row', '__source_file__', 'alpha', 'f1_target', 'f2_target',
       'f3_target', 'f4_target', 'f5_target', 'weight_label'],
      dtype='object')

In [26]:
# Assumindo `df` e `pd` já existem no notebook

# =========================================
# 1) Preparação + gap_pct + remoção outliers
# =========================================
base = df.copy()

# Corrige possível typo de schema
if "weight_label" not in base.columns and "weigh_label" in base.columns:
    base = base.rename(columns={"weigh_label": "weight_label"})
elif "weight_label" in base.columns and "weigh_label" in base.columns:
    base["weight_label"] = base["weight_label"].fillna(base["weigh_label"])
    base = base.drop(columns=["weigh_label"])

required_cols = ["instancia", "alpha", "clientes", "classe", "weight_label", "gap"]
missing = [c for c in required_cols if c not in base.columns]
if missing:
    raise ValueError(f"Colunas ausentes: {missing}")

dados = base[required_cols].copy()

dados["alpha"] = pd.to_numeric(dados["alpha"], errors="coerce")
dados["clientes"] = pd.to_numeric(dados["clientes"], errors="coerce")
dados["classe"] = pd.to_numeric(dados["classe"], errors="coerce")
dados["gap"] = pd.to_numeric(dados["gap"], errors="coerce")

dados = dados.dropna(subset=["alpha", "clientes", "classe", "weight_label", "gap"])
dados["clientes"] = dados["clientes"].astype(int)
dados["classe"] = dados["classe"].astype(int).astype(str)
dados["weight_label"] = dados["weight_label"].astype(str)

# Regra: gap negativo vira zero
dados["gap"] = dados["gap"].clip(lower=0)

# Gap em percentual (sempre gap * 100)
dados["gap_pct"] = dados["gap"] * 100

# Remove extremos por grupo (IQR por weight_label)
iqr_stats = (
    dados.groupby("weight_label")["gap_pct"]
    .quantile([0.25, 0.75]).unstack()
    .rename(columns={0.25: "q1", 0.75: "q3"})
)
iqr_stats["iqr"] = iqr_stats["q3"] - iqr_stats["q1"]
iqr_stats["lim_inf"] = iqr_stats["q1"] - 1.5 * iqr_stats["iqr"]
iqr_stats["lim_sup"] = iqr_stats["q3"] + 1.5 * iqr_stats["iqr"]

dados_aux = dados.join(iqr_stats[["lim_inf", "lim_sup"]], on="weight_label")
dados_sem_extremos = dados_aux[
    (dados_aux["gap_pct"] >= dados_aux["lim_inf"]) &
    (dados_aux["gap_pct"] <= dados_aux["lim_sup"])
].drop(columns=["lim_inf", "lim_sup"])

# =========================================
# 2) Resumo + linha All instances
# =========================================
resumo_cls = (
    dados_sem_extremos
    .groupby(["alpha", "clientes", "classe", "weight_label"], as_index=False)
    .agg(gap_medio_pct=("gap_pct", "mean"))
)

resumo_all = (
    dados_sem_extremos
    .groupby(["alpha", "clientes", "weight_label"], as_index=False)
    .agg(gap_medio_pct=("gap_pct", "mean"))
)
resumo_all["classe"] = "all"

resumo = pd.concat([resumo_cls, resumo_all], ignore_index=True)

# Linhas: clientes, classe | Colunas: alpha, weight_label
tabela_media = resumo.pivot_table(
    index=["clientes", "classe"],
    columns=["alpha", "weight_label"],
    values="gap_medio_pct",
    aggfunc="mean"
)

display(tabela_media.round(3))

# =========================================
# 3) Função para gerar LaTeX
# =========================================
def gap_table_to_latex_alpha_weight(
    tabela_media: pd.DataFrame,
    clientes_order=None,
    classes_order=None,
    alpha_order=None,
    weight_order=None,
    class_roman_map=None,
    caption=None,
    label=None,
    table_env=True,
):
    tab = tabela_media.copy()

    if not isinstance(tab.index, pd.MultiIndex) or tab.index.nlevels != 2:
        raise ValueError("tabela_media precisa ter índice MultiIndex: (clientes, classe).")
    if not isinstance(tab.columns, pd.MultiIndex) or tab.columns.nlevels != 2:
        raise ValueError("tabela_media precisa ter colunas MultiIndex: (alpha, weight_label).")

    tab.index = tab.index.set_names(["clientes", "classe"])
    tab.columns = tab.columns.set_names(["alpha", "weight_label"])

    # Helpers definidos antes do uso (corrige UnboundLocalError)
    def num_key(x):
        try:
            return (0, float(x))
        except Exception:
            return (1, str(x))

    def weight_key(w):
        s = str(w)
        digits = "".join(ch for ch in s if ch.isdigit())
        return (0, int(digits)) if digits else (1, s)

    if clientes_order is None:
        clientes_order = sorted(tab.index.get_level_values("clientes").unique().tolist(), key=num_key)

    raw_classes = tab.index.get_level_values("classe").unique().tolist()
    has_all = any(str(c).lower() == "all" for c in raw_classes)

    if classes_order is None:
        sem_all = [c for c in raw_classes if str(c).lower() != "all"]
        classes_order = sorted(sem_all, key=num_key)
        if has_all:
            classes_order.append("all")
    else:
        classes_order = list(classes_order)
        if has_all and not any(str(c).lower() == "all" for c in classes_order):
            classes_order.append("all")

    if alpha_order is None:
        alpha_order = sorted(tab.columns.get_level_values("alpha").unique().tolist(), key=num_key)
    if weight_order is None:
        weight_order = sorted(tab.columns.get_level_values("weight_label").unique().tolist(), key=weight_key)

    if class_roman_map is None:
        class_roman_map = {"1": "I", "2": "II", "3": "III", "4": "IV", 1: "I", 2: "II", 3: "III", 4: "IV"}

    full_rows = pd.MultiIndex.from_product(
        [clientes_order, classes_order], names=["clientes", "classe"]
    )
    full_cols = pd.MultiIndex.from_product(
        [alpha_order, weight_order], names=["alpha", "weight_label"]
    )
    mat = tab.reindex(index=full_rows, columns=full_cols)

    def alpha_label(a):
        try:
            return rf"$\alpha={float(a):g}$"
        except Exception:
            return rf"$\alpha={a}$"

    def class_label(cl):
        if str(cl).lower() == "all":
            return "All instances"
        return class_roman_map.get(cl, class_roman_map.get(str(cl), str(cl)))

    def clients_label(c):
        try:
            return f"{int(float(c))} clients"
        except Exception:
            return f"{c} clients"

    def fmt_pct(v):
        if pd.isna(v):
            return "-"
        return f"{v:.1f}".replace(".", ",") + r"\%"

    A = len(alpha_order)
    W = len(weight_order)
    colspec = "ll" + "c" * (A * W)

    lines = []
    if table_env:
        lines += [r"\begin{table}[t]", r"\centering"]
    if caption:
        lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")

    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")

    # Header 1: blocos por alpha
    row1 = [r"\multicolumn{1}{c}{\textbf{}}", r"\multicolumn{1}{c}{\textbf{}}"]
    row1 += [rf"\multicolumn{{{W}}}{{c}}{{\textbf{{{alpha_label(a)}}}}}" for a in alpha_order]
    lines.append(" & ".join(row1) + r" \\")

    # cmidrule por alpha
    cmid = []
    start = 3
    for _ in alpha_order:
        end = start + W - 1
        cmid.append(rf"\cmidrule(lr){{{start}-{end}}}")
        start = end + 1
    lines.append(" ".join(cmid))

    # Header 2: pesos
    row2 = [r"\multicolumn{1}{c}{\textbf{Clientes}}", r"\multicolumn{1}{c}{\textbf{Classe}}"]
    for _a in alpha_order:
        for w in weight_order:
            row2.append(rf"\multicolumn{{1}}{{c}}{{\textbf{{{w}}}}}")
    lines.append(" & ".join(row2) + r" \\")
    lines.append(r"\midrule")

    # Corpo: linhas por clientes e classe (incluindo all)
    for i, c in enumerate(clientes_order):
        rows_c = [(c, cl) for cl in classes_order]
        nrows = len(rows_c)

        for j, (_, cl) in enumerate(rows_c):
            row = [rf"\multirow{{{nrows}}}{{*}}{{\textbf{{{clients_label(c)}}}}}"] if j == 0 else [""]
            row.append(class_label(cl))

            for a in alpha_order:
                for w in weight_order:
                    row.append(fmt_pct(mat.loc[(c, cl), (a, w)]))

            lines.append(" & ".join(row) + r" \\")

        if i != len(clientes_order) - 1:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    if table_env:
        lines.append(r"\end{table}")

    return "\n".join(lines)

# =========================================
# 4) Gerar LaTeX
# =========================================
latex_code = gap_table_to_latex_alpha_weight(
    tabela_media,
    caption="Average gap (\\%) by clients/class (rows) and $\\alpha$/weight (columns), without outliers.",
    label="tab:gap_alpha_weight",
    table_env=True,
)

print(latex_code)

alpha              0.01                                            0.99  \
weight_label    $w_{1}$ $w_{2}$ $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$ $w_{1}$   
clientes classe                                                           
5        1        1.572   1.605   4.854   2.389   2.411   5.154   0.018   
         2        1.769   1.840   3.326   2.278   1.554   3.308   0.029   
         3        2.971   2.755   5.347   3.091   2.862   5.543   0.015   
         4        6.820   6.427  11.851   9.434   6.177  12.842   0.644   
         all      3.051   2.942   5.971   3.955   3.053   6.302   0.145   
10       1       52.815  62.387  68.041  61.489  54.586  65.606  41.462   
         2       53.722  63.155  68.220  62.679  52.323  65.099  45.727   
         3       52.005  58.649  65.343  63.397  52.310  64.973  38.005   
         4       63.859  66.866  68.992  65.911  59.790  74.589  56.726   
         all     55.600  62.764  67.649  63.369  54.752  67.567  45.480   

alpha                                                    
weight_label    $w_{2}$ $w_{3}$ $w_{4}$ $w_{5}$ $w_{6}$  
clientes classe                                          
5        1        0.014   0.196   0.105   0.014   0.541  
         2        0.012   0.016   0.442   0.052   0.381  
         3        1.059   0.146   0.191   0.304   1.240  
         4        0.587   0.908   0.123   0.987   0.389  
         all      0.411   0.276   0.222   0.296   0.658  
10       1       38.194  67.305  61.900  50.541  67.468  
         2       36.496  62.844  55.145  43.247  65.466  
         3       38.119  67.099  61.798  46.595  64.624  
         4       49.534  78.261  70.049  62.317  78.022  
         all     40.586  68.877  62.223  50.675  68.895

\begin{table}[t]
\centering
\caption{Average gap (\%) by clients/class (rows) and $\alpha$/weight (columns), without outliers.}
\label{tab:gap_alpha_weight}
\begin{tabular}{llcccccccccccc}
\toprule
\multicolumn{1}{c}{\textbf{}} & \multicolumn{1}{c}{\textbf{}} & \multicolumn{6}{c}{\textbf{$\alpha=0.01$}} & \multicolumn{6}{c}{\textbf{$\alpha=0.99$}} \\
\cmidrule(lr){3-8} \cmidrule(lr){9-14}
\multicolumn{1}{c}{\textbf{Clientes}} & \multicolumn{1}{c}{\textbf{Classe}} & \multicolumn{1}{c}{\textbf{$w_{1}$}} & \multicolumn{1}{c}{\textbf{$w_{2}$}} & \multicolumn{1}{c}{\textbf{$w_{3}$}} & \multicolumn{1}{c}{\textbf{$w_{4}$}} & \multicolumn{1}{c}{\textbf{$w_{5}$}} & \multicolumn{1}{c}{\textbf{$w_{6}$}} & \multicolumn{1}{c}{\textbf{$w_{1}$}} & \multicolumn{1}{c}{\textbf{$w_{2}$}} & \multicolumn{1}{c}{\textbf{$w_{3}$}} & \multicolumn{1}{c}{\textbf{$w_{4}$}} & \multicolumn{1}{c}{\textbf{$w_{5}$}} & \multicolumn{1}{c}{\textbf{$w_{6}$}} \\
\midrule
\multirow{5}{*}{\textbf{5 clients}} & I & 1,6\% & 1,